#1 — Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, when, lit, trim
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

In [0]:
#read bronze table 
source_table = "company_risk_intelligence_platform.bronze.yf_news"

df = spark.read.table(source_table)

display(df.limit(5))

In [0]:
df.printSchema()

# 2 — Silver Transformation (Flatten + Clean + Safe Casting)
Key design:
- Flatten content struct
- Extract nested fields (provider, urls, thumbnail)
- Safe timestamp parsing (NO cast errors)
- Keep only useful silver columns
- Deduplicate on id

In [0]:
def transform_yf_news(df):

    # -----------------------------
    # 1. Flatten struct "content"
    # -----------------------------
    df = df.select(
        col("id").alias("news_id"),
        col("content.title").alias("title"),
        col("content.description").alias("description"),
        col("content.summary").alias("summary"),
        col("content.storyline").alias("storyline"),
        col("content.contentType").alias("content_type"),

        col("content.pubDate").alias("pub_date_raw"),
        col("content.displayTime").alias("display_time_raw"),

        col("content.provider.displayName").alias("provider_name"),
        col("content.provider.sourceId").alias("provider_source"),
        col("content.provider.url").alias("provider_url"),

        col("content.canonicalUrl.url").alias("canonical_url"),
        col("content.clickThroughUrl.url").alias("clickthrough_url"),

        col("content.thumbnail.originalUrl").alias("thumbnail_url"),

        col("content.isHosted").alias("is_hosted"),
        col("content.metadata.editorsPick").alias("is_editors_pick"),

        col("ingestion_ts"),
        col("file_path")
    )

    # -----------------------------
    # 2. SAFE timestamp conversion
    # (fixes empty string crash)
    # -----------------------------
    df = df.withColumn(
        "pub_date",
        to_timestamp(when(trim(col("pub_date_raw")) == "", None).otherwise(col("pub_date_raw")))
    ).withColumn(
        "display_time",
        to_timestamp(when(trim(col("display_time_raw")) == "", None).otherwise(col("display_time_raw")))
    )

    # -----------------------------
    # 3. Drop raw timestamp strings
    # -----------------------------
    df = df.drop("pub_date_raw", "display_time_raw")

    # -----------------------------
    # 4. Deduplicate (SCD1 logic relies on clean keys)
    # -----------------------------
    df = df.dropDuplicates(["news_id"])

    return df

#3 — SCD Type 1 Merge Function

In [0]:
def scd_merge_table(spark, source_df, target_table, business_key):

    if not spark.catalog.tableExists(target_table):

        print("First Load: Creating Silver Table ->", target_table)

        source_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

        print("Table Created")

    else:

        print("Incremental Load: Performing SCD Type 1 Merge")

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )

        delta_table.alias("target") \
            .merge(
                source_df.alias("source"),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

        print("Merge Completed")

#4 — MAIN SILVER PIPELINE (SCD1 Execution)

In [0]:
source_table = "company_risk_intelligence_platform.bronze.yf_news"
target_table = "company_risk_intelligence_platform.silver.yf_news"

business_key = ["news_id"]

df = spark.read.table(source_table)

silver_df = transform_yf_news(df)

print("Silver Record Count:", silver_df.count())

scd_merge_table(
    spark,
    silver_df,
    target_table,
    business_key
)

Silver Layer Transformations – Yahoo Finance News

- Flatten nested JSON structure into analytics-friendly columns.
- Extract article metadata, publisher information, URLs, and premium content indicators.
- Convert publication and display timestamps to Spark timestamp format.
- Remove technical ingestion fields not required for analytics.
- Deduplicate records using news_id.
- Load data into the Silver layer using SCD Type 1 merge, where existing articles are updated and new articles are inserted.
- You’ve reached the Free limit for chats with attachments
- Upgrade now or wait until tomorrow at 1:45 AM to keep using files, or chat now without files.

This Silver transformation flattens Yahoo Finance news data by extracting nested fields from the content struct. It standardizes timestamps using safe parsing to avoid cast failures from malformed or empty values. The pipeline deduplicates records using news_id and applies SCD Type 1 logic to ensure the latest version of each news article is always reflected in the Silver layer, overwriting prior records without maintaining history.